# TRIBE v2 — Colab Backend

**Instructions:**
1. Run **Cell 1** — installs numpy correctly
2. **Restart runtime** (`Runtime → Restart runtime` or `Kernel → Restart Kernel`)
3. Run **Cells 2 through 7** top to bottom
4. Paste the public URL into the frontend Backend field

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Run this, then RESTART RUNTIME, then run Cell 2+
# ═══════════════════════════════════════════════════════════

# numba 0.61+ supports numpy 2.x — install it first to prevent conflicts
!pip install -q 'numba>=0.61.0'
# Pin numpy to exactly what tribev2 requires
!pip install -q 'numpy==2.2.6' --force-reinstall

import numpy as np
print(f'numpy: {np.__version__}')
print()
print('✓ Done. NOW RESTART THE RUNTIME, then continue from Cell 2.')
print('  Colab  → Runtime menu → Restart runtime')
print('  VSCode → Kernel toolbar → Restart')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Install all dependencies (after restart)
# ═══════════════════════════════════════════════════════════
import numpy as np, sys
assert np.__version__ == '2.2.6', (
    f'numpy is {np.__version__}, need 2.2.6. Did you restart after Cell 1?'
)
print(f'numpy {np.__version__} ✓')

# System deps
!apt-get install -y -q ffmpeg 2>/dev/null || echo 'ffmpeg skipped'

# torch — skip reinstall if already 2.x (Colab ships with it)
import subprocess
r = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'torch {r.stdout.strip()} ✓ (skipping reinstall)')
else:
    print('Installing torch...')
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Clone tribev2 before installing it
import os
clone_dir = '/content/tribev2'
if not os.path.exists(clone_dir):
    !git clone -q https://github.com/facebookresearch/tribev2 {clone_dir}
    print('tribev2 cloned')
else:
    !git -C {clone_dir} pull -q
    print('tribev2 updated')

# Install tribev2 WITHOUT deps so pip can't clobber numpy
!pip install -q -e {clone_dir} --no-deps

# Install tribev2's actual deps manually (excluding numpy/torch which are already correct)
!pip install -q einops omegaconf x-transformers huggingface_hub \
    moviepy soundfile gtts spacy langdetect Levenshtein \
    transformers accelerate

# whisperx — needs to come after torch
!pip install -q whisperx

# Backend deps
!pip install -q fastapi 'uvicorn[standard]' yt-dlp pydantic nest_asyncio

# Re-pin numpy LAST — some packages above may have tried to downgrade it
!pip install -q 'numpy==2.2.6' --force-reinstall

# Final verification
import numpy as np, torch, tribev2
print()
print(f'numpy  : {np.__version__}  (need 2.2.6)')
print(f'torch  : {torch.__version__}')
print(f'tribev2: {tribev2.__file__}')
from tribev2 import TribeModel
print(f'TribeModel ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Clone / update brain-neuro backend
# ═══════════════════════════════════════════════════════════
import os

repo_dir = '/content/brain-neuro'
if os.path.exists(repo_dir):
    !git -C {repo_dir} pull -q
    print('brain-neuro updated')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git {repo_dir}
    print('brain-neuro cloned')

os.chdir(repo_dir)
print(f'cwd: {os.getcwd()}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — HuggingFace login (model weights auto-download on first run)
# ═══════════════════════════════════════════════════════════
import huggingface_hub

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('Using HF_TOKEN from Colab secrets')
except Exception:
    hf_token = input('Enter your HuggingFace token (from huggingface.co/settings/tokens): ')

huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('HuggingFace login ✓')
print('Model weights will download automatically on first /analyze request.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 (Optional) — Upload cookies.txt for Instagram/TikTok
# Skip if only using YouTube.
# ═══════════════════════════════════════════════════════════
import shutil, os

try:
    from google.colab import files
    print('Upload your cookies.txt (or skip this cell for YouTube only)')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, 'cookies.txt')
        print('cookies.txt saved ✓')
except ImportError:
    if os.path.exists('cookies.txt'):
        print('cookies.txt found ✓')
    else:
        print('No cookies.txt — place it next to backend.py if needed')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Install cloudflared tunnel
# ═══════════════════════════════════════════════════════════
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Start backend + public tunnel
# ═══════════════════════════════════════════════════════════
import nest_asyncio, uvicorn, threading, subprocess, time, re, os

os.chdir('/content/brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)
print('FastAPI running on :8000 ✓')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('Waiting for tunnel...')
for line in proc.stdout:
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print()
        print('═' * 50)
        print(f'  BACKEND URL: {url}')
        print('═' * 50)
        print('→ Paste this into the frontend Backend field')
        break